<h1 align=center style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Fraud Detection!
</font>
</h1>

<h2 dir="ltr" align="left" style="line-height:200%; font-family:vazir; color:#0099cc">
<font face="vazir" color="#0099cc">
Dataset Description
</font>
</h2>

<p dir="rtl" style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
</font>
</p>

<p dir=rtl style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
Import libraries and load data from the <code>Data</code> folder.
</font>
</p>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('Data/train.csv')
test = pd.read_csv('Data/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'\nFraud distribution:\n{train["is_fraud"].value_counts()}')
print(f'\nFraud rate: {train["is_fraud"].mean():.4f}')

<h2 dir="ltr" align="left" style="line-height:200%; font-family:vazir; color:#0099cc">
<font face="vazir" color="#0099cc">
Preprocessing and Feature Engineering
</font>
</h2>

<p dir="rtl" style="direction: rtl; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size="3">
In this task, you are free to apply any preprocessing and/or feature engineering techniques of your choice.
<br>
The specific techniques you use will <b>not</b> be evaluated directly by the grading system. Instead, their impact will be reflected in your model's performance. Therefore, the better your preprocessing and feature engineering methods are in improving the model's accuracy, the higher the score you are likely to achieve for this task.
You may also set aside a portion of the available data for validation purposes in this section.
</font>
</p>

In [ ]:
def parse_timestamp(ts):
    parts = ts.split(':')
    mins = int(parts[0])
    sec_parts = parts[1].split('.')
    secs = int(sec_parts[0])
    frac = int(sec_parts[1])
    return mins * 60 + secs + frac / 10.0

def build_features(df, batch_fraud_rates=None, is_train=True):
    df = df.copy()

    # Timestamp features
    df['timestamp_seconds'] = df['timestamp'].apply(parse_timestamp)
    df['ts_minute'] = df['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df['ts_second'] = df['timestamp'].apply(lambda x: float(x.split(':')[1]))
    df['ts_sin_min'] = np.sin(2 * np.pi * df['ts_minute'] / 60)
    df['ts_cos_min'] = np.cos(2 * np.pi * df['ts_minute'] / 60)

    # Batch-level aggregated features
    batch_agg = df.groupby('processing_batch_id').agg(
        batch_size=('user_id', 'count'),
        batch_amount_mean=('transaction_amount', 'mean'),
        batch_speed_mean=('transaction_speed_seconds', 'mean'),
        batch_ip_mean=('ip_risk_score', 'mean'),
        batch_age_mean=('user_age_days', 'mean'),
        batch_amount_median=('transaction_amount', 'median'),
        batch_speed_median=('transaction_speed_seconds', 'median'),
        batch_ip_median=('ip_risk_score', 'median'),
    ).reset_index()
    batch_std = df.groupby('processing_batch_id')['transaction_amount'].std().fillna(0).reset_index()
    batch_std.columns = ['processing_batch_id', 'batch_amount_std']
    batch_agg = batch_agg.merge(batch_std, on='processing_batch_id', how='left')
    df = df.merge(batch_agg, on='processing_batch_id', how='left')

    # Deviation from batch mean
    df['amount_dev_from_batch'] = df['transaction_amount'] - df['batch_amount_mean']
    df['speed_dev_from_batch'] = df['transaction_speed_seconds'] - df['batch_speed_mean']
    df['ip_dev_from_batch'] = df['ip_risk_score'] - df['batch_ip_mean']
    df['age_dev_from_batch'] = df['user_age_days'] - df['batch_age_mean']

    # Rank within batch
    df['amount_batch_rank'] = df.groupby('processing_batch_id')['transaction_amount'].rank(pct=True)
    df['speed_batch_rank'] = df.groupby('processing_batch_id')['transaction_speed_seconds'].rank(pct=True)
    df['ip_batch_rank'] = df.groupby('processing_batch_id')['ip_risk_score'].rank(pct=True)
    df['age_batch_rank'] = df.groupby('processing_batch_id')['user_age_days'].rank(pct=True)

    # Ratio features
    df['amount_per_age'] = df['transaction_amount'] / (df['user_age_days'] + 1)
    df['ip_per_speed'] = df['ip_risk_score'] / (df['transaction_speed_seconds'] + 0.1)
    df['amount_per_speed'] = df['transaction_amount'] / (df['transaction_speed_seconds'] + 0.1)

    # Interaction features
    df['ip_x_amount'] = df['ip_risk_score'] * df['transaction_amount']
    df['ip_x_speed'] = df['ip_risk_score'] * df['transaction_speed_seconds']
    df['amount_x_speed'] = df['transaction_amount'] * df['transaction_speed_seconds']
    df['ip_x_age'] = df['ip_risk_score'] * df['user_age_days']
    df['speed_x_age'] = df['transaction_speed_seconds'] * df['user_age_days']

    # Power / log features
    df['ip_risk_score_sq'] = df['ip_risk_score'] ** 2
    df['user_age_days_log'] = np.log1p(df['user_age_days'])
    df['transaction_amount_log'] = np.log1p(df['transaction_amount'])
    df['transaction_speed_log'] = np.log1p(df['transaction_speed_seconds'])

    # Binned features
    df['ip_risk_bin'] = pd.cut(df['ip_risk_score'], bins=[0, 30, 50, 70, 85, 100], labels=False)
    df['speed_bin'] = pd.cut(df['transaction_speed_seconds'], bins=[0, 5, 10, 15, 20, 25], labels=False)
    df['age_bin'] = pd.cut(df['user_age_days'], bins=[0, 30, 90, 200, 500, 1000], labels=False)
    df['amount_bin'] = pd.cut(df['transaction_amount'], bins=[0, 50, 200, 500, 1000, 12000], labels=False)

    # Batch fraud rate (target encoding)
    if is_train:
        batch_fraud_rates = df.groupby('processing_batch_id')['is_fraud'].mean().to_dict()
        df['batch_fraud_rate'] = df['processing_batch_id'].map(batch_fraud_rates)
    else:
        df['batch_fraud_rate'] = df['processing_batch_id'].map(batch_fraud_rates).fillna(0.26)

    # Category frequency encoding
    for col in ['product_category', 'payment_method']:
        freq_map = df[col].value_counts(normalize=True).to_dict()
        df[f'{col}_freq'] = df[col].map(freq_map)

    # One-hot encoding
    df = pd.get_dummies(df, columns=['product_category', 'payment_method'], drop_first=False)

    return df, batch_fraud_rates

train_feat, batch_fraud_rates = build_features(train, is_train=True)
test_feat, _ = build_features(test, batch_fraud_rates=batch_fraud_rates, is_train=False)

# Align columns
train_cols = set(train_feat.columns)
test_cols = set(test_feat.columns)
for c in train_cols - test_cols:
    if c != 'is_fraud':
        test_feat[c] = 0
for c in test_cols - train_cols:
    train_feat[c] = 0

print(f'Train features shape: {train_feat.shape}')
print(f'Test features shape: {test_feat.shape}')

drop_cols = ['timestamp', 'user_id', 'is_fraud']
feature_cols = [c for c in train_feat.columns if c not in drop_cols]
print(f'Number of features: {len(feature_cols)}')

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Training model
</font>
</h2>



In [ ]:
X = train_feat[feature_cols].values
y = train_feat['is_fraud'].values
X_test = test_feat[feature_cols].values

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]

    scale_pos = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)

    model = XGBClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        scale_pos_weight=scale_pos,
        random_state=42 + fold,
        eval_metric='logloss'
    )

    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    val_pred = model.predict(X_val)
    fold_acc = accuracy_score(y_val, val_pred)
    fold_accuracies.append(fold_acc)
    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / n_folds

    print(f'Fold {fold+1} Accuracy: {fold_acc:.6f}')

print(f'\nMean CV Accuracy: {np.mean(fold_accuracies):.6f} (+/- {np.std(fold_accuracies):.6f})')

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
evaluation metric
</font>
</h2>



In [ ]:
from sklearn.metrics import accuracy_score

print('OOF Accuracy:', accuracy_score(y, oof_preds))
print('\nClassification Report:')
print(classification_report(y, oof_preds))

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
prediction for test dataset
</font>
</h2>


In [ ]:
test_final_preds = (test_preds >= 0.5).astype(int)

submission = pd.DataFrame({
    'user_id': test['user_id'],
    'is_fraud': test_final_preds
})

print(f'Submission shape: {submission.shape}')
print(f'Predicted fraud rate: {submission["is_fraud"].mean():.4f}')
print(submission.head(10))

<h2 dir="ltr" align="left" style="line-height:200%;font-family:vazir;color:#0099cc">

<font face="vazir" color="#0099cc">

<b>Submission Cell</b>

</font>

</h2>

<p dir="ltr" style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">

<font face="vazir" size="3">
    Run the following cell to generate the <code>result.zip</code> file. Please make sure that you have saved all changes made to the notebook (<code>Ctrl+S</code>) before executing the cell below. Otherwise, your final competition score may be reduced to zero.
    <br>
    Additionally, if you are using Google Colab to run this notebook, make sure to download the latest version of your notebook and include it in the submitted <code>result.zip</code> file before submission.
</font>
</p>

In [ ]:
import zipfile
import joblib
import os

if not os.path.exists(os.path.join(os.getcwd(), 'fraudy.ipynb')):
    %notebook -e fraudy.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['fraudy.ipynb', 'submission.csv']
compress(file_names)